# Building a Dataset Using XArray

In the previous notebook, you learned how to make `DataArray`s with Xarray. But Xarray has another level of organization, referred to as a `Dataset`, which is a collection of DataArrays.
  - The `DataArray`: A multidimensional array with labels describing each of its dimensions.
  - The `Dataset`: A collection of `DataArrays` and their relationships.

We'll be working with a good amount of data organized in `Xarray`, mainly to load data, select the variables we're interested in, and convert the data into the structure we want, so let's take a little time to familiarize ourselves with some of these features

## Setup

### Import Libraries

In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt

## Demos: Creating a Dataset from DataArrays

You already know how to create an `xr.DataArray`. A `Dataset` combines several named DataArrays into one object. **Dimensions** describe the axes of each variable; **coordinates** provide labels for positions along those axes. Variables can share dimensions and coordinates without having the same shape.

Let's build a miniature behavioral experiment before examining the real recording.

**Example:** We measured reaction time and stimulus contrast over four trials. Create two DataArrays using the **same** `trial` coordinate.

In [ ]:
trials = [10, 11, 12, 13]

reaction_time = xr.DataArray(
    [0.42, 0.35, 0.51, 0.39],
    dims=['trial'],
    coords={'trial': trials},
    name='reaction_time',
)

contrast_left = xr.DataArray(
    [0.0, 0.5, 1.0, 0.5],
    dims=['trial'],
    coords={'trial': trials},
    name='contrast_left',
)

reaction_time

<xarray.DataArray 'reaction_time' (trial: 4)> Size: 32B
array([0.42, 0.35, 0.51, 0.39])
Coordinates:
  * trial    (trial) int64 32B 10 11 12 13

**Example:** Combine the DataArrays into a Dataset. Dictionary keys become the variable names. Notice that `trial` appears **once** as a shared coordinate, while both measurements are separate data variables.

In [ ]:
experiment = xr.Dataset({
    'reaction_time': reaction_time,
    'contrast_left': contrast_left,
})
experiment

<xarray.Dataset> Size: 96B
Dimensions:        (trial: 4)
Coordinates:
  * trial          (trial) int64 32B 10 11 12 13
Data variables:
    reaction_time  (trial) float64 32B 0.42 0.35 0.51 0.39
    contrast_left  (trial) float64 32B 0.0 0.5 1.0 0.5

**Example:** Add a `response_type` DataArray with the values `['left', 'right', 'right', 'left']`, indexed by the same trials. Then add it to the Dataset and inspect `experiment`.

*Hint:* A Dataset can be extended just like a dictionary: `experiment['response_type'] = ...`.

In [ ]:
response_type = xr.DataArray(
    ['left', 'right', 'right', 'left'],
    dims=['trial'],
    coords={'trial': trials},
)
experiment['response_type'] = response_type
experiment

<xarray.Dataset> Size: 176B
Dimensions:        (trial: 4)
Coordinates:
  * trial          (trial) int64 32B 10 11 12 13
Data variables:
    reaction_time  (trial) float64 32B 0.42 0.35 0.51 0.39
    contrast_left  (trial) float64 32B 0.0 0.5 1.0 0.5
    response_type  (trial) <U5 80B 'left' 'right' 'right' 'left'

### Different dimensions, same Dataset

Suppose we also measured pupil size at three time points during each trial. This variable has **two dimensions**, `trial` and `time`. The other variables only have `trial`. They can all live in the same Dataset because the shared `trial` axis refers to the same trials.

**Example:** Create the pupil-size DataArray and add it to `experiment`. Examine the Dataset's dimensions, coordinates, and data variables.

In [ ]:
pupil_size = xr.DataArray(
    [[2.1, 2.3, 2.2],
     [2.0, 2.2, 2.4],
     [2.3, 2.4, 2.5],
     [2.2, 2.1, 2.3]],
    dims=['trial', 'time'],
    coords={'trial': trials, 'time': [0.0, 0.5, 1.0]},
)
experiment['pupil_size'] = pupil_size
experiment

**Exercise:** Select trial `12` from the *whole Dataset* using `.sel(trial=12)`. Which measurements are returned? Then select the pupil measurements at time `0.5` for **all trials**.

Notice that coordinate labels (such as trial `12`) are not necessarily the same as positional indices.

In [ ]:
experiment.sel(trial=12)

**Key idea:** Matching dimension names tell Xarray which axes are related; coordinate labels tell it *which observations correspond*. If labels differ, Xarray may reorder, align, or introduce missing values, depending on the operation. Always inspect the coordinates when combining data.

Now look for these same relationships in the real Steinmetz Dataset below: some variables describe trials, some describe cells, and `spike_count` spans cells, trials, and time.

## Working with XArray Datasets

Datasets describe the relationships between multiple data variables, often connected by their coordinates.  For example, there could be different behavioral measurements taken at the same time points, or different data might be related to the same subject.  `xr.Dataset` objects help in this way to describe a more-complete picture of an experiment.

Let's explore Datasets by looking at the data from a session of the [Steinmetz et al, 2019 paper](https://www.nature.com/articles/s41586-019-1787-x). In this notebook, we'll be exploring the spike counts data recorded during a specific session and investigate differences between brain regions and their relation to behavioral, task, or performance variables.  This data, stored in the netCDF format (the `.nc` extension), can be loaded as an XArray Dataset.

| Code | Description |
|---|---|
| `xr.load_dataset("file.nc")` | Load a netCDF file as an XArray Dataset. |
| `dset['variable_name']` | Access a specific variable (DataArray) from the dataset. |
| `dset[['var1', 'var2']]` | Access multiple variables from the dataset simultaneously. |
| `dset['variable'].loc[start:end]` | Select data using location-based indexing for specific range. |
| `sum(boolean_array).item()` | Count True values in boolean array and extract as scalar. |
| `len(array)` | Get the length of an array. |

Run the following code cell to load the dataset into Python using XArray. Explore the dataset. Note that different variables are related to different sets of coordinates:

#### **Exercises**

In [ ]:
dset = xr.load_dataset(f"data/steinmetz_2017-12-05_Lederberg.nc")
dset

**Example**: Get the reaction time for each trial in the session.

In [ ]:
dset['reaction_time']

**Exercise**: Get the response type for each trial:

**Exercise**: Get the contrast levels of both the left and right stimulus for each trial (note: you'll need double-square brackets for this (e.g. `dset[['a', 'b']]`)).

**Exercise**: Get the response times for trials 10 through 16 (tip: `.loc`).

## Spike Count Analysis For Specific Brain Regions: Practicing Logical Indexing on Multidimensional Numpy Arrays

Different areas in the brain are commonly believed to specialize in specific aspects of behavior or task. This makes analyzing spiking activity across brain regions an essential step in many neuroscience data analysis pipelines. Over the next couple of exercises let's focus on how the spike counts of neurons differ across different brain regions.

| Code | Description |
|---|---|
| `np.unique(data)` | Finds the unique elements of an array. |
| `bool_array = var_name == specific_value` | Creates a boolean array where each element is `True` if the corresponding element in `var_name` equals `specific_value`; otherwise `False`. |
| `var_name[bool_array]` | Uses a boolean array to filter elements of `var_name`, selecting only those that correspond to `True` in the boolean array. |
| `var_name.mean(axis=0)` or `np.mean(var_name, axis=0)`| Compute the mean across the first dimension, i.e. across rows. |
| `var_name.mean(axis=(0, 1))` or `np.mean(var_name, axis=(0, 1))` | Compute the mean across the first and second dimensions |
| `var_name.sum(axis=N)` | Sum along axis N (0=rows, 1=columns, etc.). |
| `array.values` | Get the underlying numpy array from an XArray object. |
| `array.item()` | Extract a scalar value from a single-element array. |
| `plt.imshow(data)` | For a 2D array, make a heat map. |
| `plt.plot(x, y)` | Make a line plot. |
| `plt.xticks(positions, labels)` | Customize x-axis tick positions and labels. |
| `data.sum(dim='name')` | Sum along named dimension in XArray. |
| `data.plot.imshow()`, `data.plot.line()` | Built-in XArray plotting methods. |

For the exercises below, we'll use the `spike_count` and `brain_groups` variables and their associated coordinate data to explore some of the spiking data.

#### **Exercises**

**Example**: What is the total number of neurons recorded from the `visual cortex`?

In [ ]:
sum(dset['brain_groups'] == 'visual cortex').item()

In [ ]:
(dset['brain_groups']== 'visual cortex').sum().item()

**Exercise**: What is the total number of neurons recorded from `thalamus`?

**Exercise**: What is the proportion of neurons recorded from `hippocampus`?

**Exercise**: How many non-zero spike count measurements were there?

**Exercise**: What proportion of cells had more than 300 spikes measured across all trials and time?

**Exercise**: (extra): What proportion of trials had more than 6000 spikes across all cells measured?

**Exercise**: Compute the population spike count (i.e. average spike count across all neurons) for neurons in `thalamus`, and visualize it using the `plt.imshow()` plotting function.

**Exercise**: From the spike counts, extract out all the trials and time points, but only the cells from the thalamus brain area. Plot the total spike counts across trials as a heat map.

**Exercise**: From the spike counts, extract out all the trials and time points, but only the cells from the Visual cortex brain area. Plot the total spike counts across trials as a heat map.

**Exercise**: For each trial and time point, show the total activity across all cells in the visual cortex

**Exercise**: For each trial and time point, show the total activity across all cells in the hippocampus.

**Example**: Make a line plot showing the average number of total spikes for all cells in the thalamus.

In [ ]:
mask = dset['brain_groups'] == 'thalamus'
data = dset['spike_count'][mask, :, :].sum(axis=0).mean(axis=0)
plt.plot(time, data);
plt.xlabel('time (secs)')
plt.ylabel('average num spikes');

**Exercise**: Make a line plot showing the average number of total spikes for all cells in the hippocampus.

**Exercise**: Make a line plot showing the average number of total spikes for all cells in the visual cortex.

**Exercise**: Make a plot showing the average spike counts for all three brain areas (visual cortex, thalamus, and hippocampus)

## Demo: Simplifying this process using XArray

#### **Exercises**

**Example**: This can also be done without creating a bunch of variables.  Here are two examples:

In [ ]:
dset['spike_count'][dset['brain_groups'] == 'visual cortex'].sum(dim='cell').plot.imshow();

In [ ]:
dset['spike_count'][dset['brain_groups'] == 'visual cortex'].sum(dim=('cell', 'trial')).plot.line();